## Data Acquisition and Profiling

In [1]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/Online_Retail.csv")

In [2]:
print(df_raw.shape)
print(df_raw.duplicated().sum())

(541909, 8)
5268


In [3]:
# Data quality report

n_rows, n_cols = df_raw.shape
date_min, date_max = df_raw['InvoiceDate'].min(), df_raw['InvoiceDate'].max()

report = {
    "n_rows": n_rows,
    "n_cols": n_cols,
    "date_range": (date_min, date_max),
    "unique_invoices": df_raw['InvoiceNo'].nunique(),
    "unique_products": df_raw['StockCode'].nunique(),
    "unique_customers_known": df_raw['CustomerID'].dropna().nunique(),
    "unique_countries": df_raw['Country'].nunique(),
    "exact_duplicates": df_raw.duplicated().sum(),
    "nulls_by_column": df_raw.isnull().sum().to_dict(),
    "cancellation_lines": df_raw['InvoiceNo'].astype(str).str.startswith('C').sum(),
    "non_positive_quantity": (df_raw['Quantity'] <= 0).sum(),
    "non_positive_price": (df_raw['UnitPrice'] <= 0).sum(),
}
for k, v in report.items():
    print(f"{k}: {v}")

n_rows: 541909
n_cols: 8
date_range: ('2010-12-01 08:26:00', '2011-12-09 12:50:00')
unique_invoices: 25900
unique_products: 4070
unique_customers_known: 4372
unique_countries: 38
exact_duplicates: 5268
nulls_by_column: {'InvoiceNo': 0, 'StockCode': 0, 'Description': 1454, 'Quantity': 0, 'InvoiceDate': 0, 'UnitPrice': 0, 'CustomerID': 135080, 'Country': 0}
cancellation_lines: 9288
non_positive_quantity: 10624
non_positive_price: 2517


In [4]:
# df_profiled

df_profiled = df_raw.copy()

df_profiled['IsCancellation'] = df_profiled['InvoiceNo'].astype(str).str.startswith('C')
df_profiled['Revenue'] = df_profiled['Quantity'] * df_profiled['UnitPrice']
df_profiled['IsKnownCustomer'] = df_profiled['CustomerID'].notna()
df_profiled['IsPositiveQty'] = df_profiled['Quantity'] > 0
df_profiled['IsPositivePrice'] = df_profiled['UnitPrice'] > 0
df_profiled['IsExactDuplicate'] = df_profiled.duplicated(keep=False)

# A row counts as a "valid purchase line" only if ALL of these hold
df_profiled['IsValidPurchaseLine'] = (
    ~df_profiled['IsCancellation']
    & df_profiled['IsKnownCustomer']
    & df_profiled['IsPositiveQty']
    & df_profiled['IsPositivePrice']
)

print(df_profiled.shape)
print(df_profiled['IsValidPurchaseLine'].value_counts())

(541909, 15)
IsValidPurchaseLine
True     397884
False    144025
Name: count, dtype: int64


In [5]:
# Rows with non-positive quantity but NOT cancellation-flagged

mystery_rows = df_profiled[
    (df_profiled['IsPositiveQty'] == False) & (~df_profiled['IsCancellation'])
]

print(mystery_rows.shape)
print(mystery_rows['StockCode'].value_counts().head(15))
print(mystery_rows[['InvoiceNo','StockCode','Description','Quantity','UnitPrice','CustomerID']].sample(15, random_state=42))

(1336, 15)
StockCode
21830     5
85175     5
22719     4
82494L    4
85172     4
72802C    4
79163     3
72807B    3
20966     3
20774     3
40016     3
23084     3
84631     3
72807A    3
85017A    3
Name: count, dtype: int64
       InvoiceNo StockCode              Description  Quantity  UnitPrice  \
292271    562548     23084                      NaN       -84        0.0   
387138    570260     35933                      NaN       -34        0.0   
478966    577122     23348  sold with wrong barcode       -57        0.0   
131708    547618     21547                      NaN       -12        0.0   
114493    546016     85172               Dotcom set      -675        0.0   
154860    549952    37482P                  Damaged       -93        0.0   
114535    546021     85175         dotcom sold sets      -345        0.0   
128467    547339    84406B                        ?      -450        0.0   
148093    549168     21161                      NaN       -10        0.0   
222036    556

In [6]:
# ---- Stock adjustment lines: internal write-offs, not customer transactions ----
# Evidence: UnitPrice == 0, CustomerID null, non-cancellation invoice,
# descriptions like "Damaged", "check", "sold as set", "?", or missing.

stock_adjustment_lines = df_profiled[
    (df_profiled['IsPositiveQty'] == False)
    & (~df_profiled['IsCancellation'])
].copy()
stock_adjustment_lines['AdjustmentReason'] = 'internal_stock_adjustment_no_customer'

print("stock_adjustment_lines:", stock_adjustment_lines.shape)
print("Confirm pattern — UnitPrice==0 share:", (stock_adjustment_lines['UnitPrice'] == 0).mean())
print("Confirm pattern — CustomerID null share:", stock_adjustment_lines['CustomerID'].isna().mean())

stock_adjustment_lines: (1336, 16)
Confirm pattern — UnitPrice==0 share: 1.0
Confirm pattern — CustomerID null share: 1.0


In [7]:
# ---- Drop exact duplicates (documented decision: data-entry error) ----

df_deduped = df_profiled.drop_duplicates(
    subset=[c for c in df_profiled.columns if c not in ['IsExactDuplicate']],
    keep='first'
).copy()
print("Before dedup:", df_profiled.shape[0], "-> After dedup:", df_deduped.shape[0])

# ---- valid_purchase_lines: RFM / clustering / prediction baseline ----

valid_purchase_lines = df_deduped[
    df_deduped['IsValidPurchaseLine']
    & (df_deduped['Country'] == 'United Kingdom')   # matches Chen et al. baseline scope
].copy()
print("valid_purchase_lines:", valid_purchase_lines.shape)

# ---- cancellation_lines: kept aside for anomaly analysis ----

cancellation_lines = df_deduped[df_deduped['IsCancellation']].copy()
print("cancellation_lines:", cancellation_lines.shape)

print(valid_purchase_lines['CustomerID'].nunique())

Before dedup: 541909 -> After dedup: 536641
valid_purchase_lines: (349203, 15)
cancellation_lines: (9251, 15)
3920


In [13]:
# Build the all-countries version of valid purchase lines (same rules as valid_purchase_lines, 
# minus the UK-only restriction) — needed for OLAP country-level analysis
all_countries_valid_lines = df_deduped[df_deduped['IsValidPurchaseLine']].copy()

# same dtype fixes as before, applied consistently
for col in ['InvoiceNo', 'StockCode', 'Description', 'Country']:
    all_countries_valid_lines[col] = all_countries_valid_lines[col].astype(str)
all_countries_valid_lines['CustomerID'] = all_countries_valid_lines['CustomerID'].astype(str)

all_countries_valid_lines.to_csv("../data/processed/all_countries_valid_purchase_lines.csv", index=False)
print(all_countries_valid_lines.shape)
print(all_countries_valid_lines['Country'].nunique(), "countries")

(392692, 15)
37 countries


In [8]:
print("Max InvoiceDate in valid_purchase_lines:", valid_purchase_lines['InvoiceDate'].max())
print("Min InvoiceDate in valid_purchase_lines:", valid_purchase_lines['InvoiceDate'].min())

# also check if that max date has a full day of transactions or just a few (partial last day)

valid_purchase_lines['InvoiceDate'] = pd.to_datetime(valid_purchase_lines['InvoiceDate'])
last_date = valid_purchase_lines['InvoiceDate'].max().normalize()
last_day_rows = valid_purchase_lines[valid_purchase_lines['InvoiceDate'].dt.normalize() == last_date]
print("Rows on the last calendar day:", last_day_rows.shape[0])
print("Time range on last day:", last_day_rows['InvoiceDate'].min(), "to", last_day_rows['InvoiceDate'].max())

Max InvoiceDate in valid_purchase_lines: 2011-12-09 12:49:00
Min InvoiceDate in valid_purchase_lines: 2010-12-01 08:26:00
Rows on the last calendar day: 430
Time range on last day: 2011-12-09 08:39:00 to 2011-12-09 12:49:00


## RFM Baseline Features

In [9]:
REFERENCE_DATE = pd.Timestamp('2011-12-10')

# ---- Build customer_features_rfm ----
# Aggregate to invoice level first (Section 15.2: "aggregate transaction lines to invoices
# before some basket-level features") — needed for a clean Frequency count and Monetary sum.

invoice_agg = valid_purchase_lines.groupby(['CustomerID', 'InvoiceNo']).agg(
    InvoiceDate=('InvoiceDate', 'min'),
    InvoiceRevenue=('Revenue', 'sum')
).reset_index()

customer_features_rfm = invoice_agg.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('InvoiceRevenue', 'sum')
).reset_index()

print(customer_features_rfm.shape)
customer_features_rfm.describe()

(3920, 4)


,CustomerID,Recency,Frequency,Monetary
count,3920.000000,3920.000000,3920.000000,3920.000000
mean,15562.059694,91.742092,4.246429,1858.424654
std,1576.594671,99.533485,7.199202,7478.631256
min,12346.000000,0.000000,1.000000,3.750000
25%,14208.750000,17.000000,1.000000,298.185000
50%,15569.500000,50.000000,2.000000,644.975000
75%,16913.250000,142.000000,5.000000,1571.285000
max,18287.000000,373.000000,209.000000,259657.300000


In [10]:
# ---- Enforce identifier dtypes, safe to re-run ----
def customerid_to_str(series):
    # Always starts from the original numeric/float representation logic,
    # regardless of whether this has already been run before.
    return series.apply(lambda x: str(int(x)) if pd.notna(x) and str(x) not in ('<NA>', 'nan') else pd.NA)

for df_ in [valid_purchase_lines, cancellation_lines, stock_adjustment_lines]:
    df_['InvoiceNo'] = df_['InvoiceNo'].astype(str)
    df_['StockCode'] = df_['StockCode'].astype(str)
    df_['CustomerID'] = customerid_to_str(df_['CustomerID'])

customer_features_rfm['CustomerID'] = customerid_to_str(customer_features_rfm['CustomerID'])

In [11]:
# ---- Enforce consistent string dtype on all text columns before saving ----
def clean_text_column(series):
    return series.astype(str).where(series.notna(), pd.NA)

text_cols_map = {
    'valid_purchase_lines': ['InvoiceNo', 'StockCode', 'Description', 'Country'],
    'cancellation_lines': ['InvoiceNo', 'StockCode', 'Description', 'Country'],
    'stock_adjustment_lines': ['InvoiceNo', 'StockCode', 'Description', 'Country', 'AdjustmentReason'],
}

tables = {
    'valid_purchase_lines': valid_purchase_lines,
    'cancellation_lines': cancellation_lines,
    'stock_adjustment_lines': stock_adjustment_lines,
}

for name, cols in text_cols_map.items():
    df_ = tables[name]
    for col in cols:
        if col in df_.columns:
            df_[col] = clean_text_column(df_[col])

In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)

valid_purchase_lines.to_csv("../data/processed/valid_purchase_lines.csv", index=False)
cancellation_lines.to_csv("../data/processed/cancellation_lines.csv", index=False)
stock_adjustment_lines.to_csv("../data/processed/stock_adjustment_lines.csv", index=False)
customer_features_rfm.to_csv("../data/processed/customer_features_rfm.csv", index=False)